# 4) Compute DTW distances and build per-case caches (dev/test)

In [ ]:
# %% [markdown]
# # 4) Compute DTW distances and build per-case caches (dev/test)

# %%
# 4.1) Imports & config
import sys, json
from pathlib import Path
import pandas as pd
import numpy as np

# Ensure project root is on PYTHONPATH
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.dtw.compute_dtw import build_cache

PAIRS_DIR = project_root / "data" / "pairs"
CACHE_DIR = project_root / "data" / "dtw_cache"

# Default DTW settings (change here if needed)
BACKEND = None            # options: None (auto), "dtaidistance", "cuda", "python"
WINDOW  = 10              # Sakoe–Chiba band half-width
N_JOBS  = None            # None -> use all logical CPUs
OVERWRITE = False         # set True to recompute everything
SHOW_PROGRESS = True

# %%
# 4.2) Compute caches for all (split, case)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

outputs = {}
for split in ("dev", "test"):
    outputs[split] = {}
    for case in ("genuine", "skilled", "random"):
        pairs_path = PAIRS_DIR / split / f"pairs_{split}_{case}.parquet"
        cache_path = CACHE_DIR / split / f"dtw_{split}_{case}.parquet"
        cache_path.parent.mkdir(parents=True, exist_ok=True)

        print(f"\n=== {split.upper()} :: {case} ===")
        print(f"pairs  → {pairs_path}")
        print(f"cache  → {cache_path}")
        build_cache(
            pairs_path=pairs_path,
            cache_path=cache_path,
            backend=BACKEND,
            window=WINDOW,
            n_jobs=N_JOBS,
            overwrite=OVERWRITE,
            show_progress=SHOW_PROGRESS,
        )
        outputs[split][case] = cache_path
        print(f"✅ wrote {cache_path}")

# %%
# 4.3) Integrity checks: cache rows must match pairs rows exactly
def _check_counts(split: str, case: str) -> None:
    ppath = PAIRS_DIR / split / f"pairs_{split}_{case}.parquet"
    cpath = CACHE_DIR / split / f"dtw_{split}_{case}.parquet"
    dfp = pd.read_parquet(ppath, engine="pyarrow")
    dfc = pd.read_parquet(cpath, engine="pyarrow")

    # counts and identity
    assert len(dfp) == len(dfc), f"{split}/{case}: cache count {len(dfc)} != pairs count {len(dfp)}"
    assert dfc["pair_id"].is_unique, f"{split}/{case}: duplicate pair_id in cache"

    # expected columns (pairwise mode)
    need = {"pair_id","label","d_raw","d_bound","path_len","len_A","len_B","backend","window","mode"}
    missing = need - set(dfc.columns)
    assert not missing, f"{split}/{case}: cache missing columns {missing}"

    # backend/window consistency
    assert dfc["backend"].nunique() == 1, f"{split}/{case}: mixed backends in one cache"
    assert dfc["window"].nunique() == 1,  f"{split}/{case}: mixed windows in one cache"

    # label present parity (for pairwise mode)
    if "label" in dfp.columns:
        assert "label" in dfc.columns, f"{split}/{case}: label missing in cache"

    # key metric invariant
    assert (dfc["d_bound"] >= dfc["d_raw"]).all(), f"{split}/{case}: found d_bound < d_raw"

    # optional: split/case columns carried through
    if {"split","case"}.issubset(dfc.columns):
        assert (dfc["split"] == split).all(), f"{split}/{case}: split column mismatch in cache"
        assert (dfc["case"]  == case).all(),  f"{split}/{case}: case column mismatch in cache"

    print(f"{split}/{case}: rows OK ({len(dfc)}) | unique pair_id OK | backend/window OK | metrics OK")

for split in ("dev", "test"):
    for case in ("genuine", "skilled", "random"):
        _check_counts(split, case)

print("\n✅ All DTW cache integrity checks passed.")

# %%
# 4.4) Quick diagnostics (optional): basic stats per case for sanity
def _summarize(split: str, case: str):
    cpath = CACHE_DIR / split / f"dtw_{split}_{case}.parquet"
    df = pd.read_parquet(cpath, engine="pyarrow")
    cols = [c for c in ["d_raw", "d_bound", "path_len", "len_A", "len_B", "d_mean", "d_min"] if c in df.columns]
    desc = df[cols].describe().T.round(3)
    try:
        display(desc)
    except NameError:
        print(desc)

for split in ("dev", "test"):
    print(f"\n=== SUMMARY: {split.upper()} ===")
    for case in ("genuine", "skilled", "random"):
        print(f"\n-- {case} --")
        _summarize(split, case)

# %%
# 4.5) Persist a tiny meta.json for downstream auditing
for split in ("dev", "test"):
    meta = {}
    for case in ("genuine", "skilled", "random"):
        cpath = CACHE_DIR / split / f"dtw_{split}_{case}.parquet"
        dfc = pd.read_parquet(cpath, engine="pyarrow")
        meta[case] = {
            "rows": int(len(dfc)),
            "mode": dfc["mode"].iloc[0] if len(dfc) else "unknown",
            "backend": dfc["backend"].iloc[0] if len(dfc) else "unknown",
            "window": int(dfc["window"].iloc[0]) if len(dfc) else WINDOW,
        }
    (CACHE_DIR / split / "meta.json").write_text(json.dumps(meta, indent=2))
    print(f"📝 wrote {CACHE_DIR / split / 'meta.json'}")